# Ollama AI Classification

In [ ]:
import requests
import pandas as pd
from pathlib import Path
print('Imports OK')

In [ ]:
df = pd.read_csv('../../data/import/clean/all-briefings.csv')
print(f'Loaded {len(df)} videos')

In [ ]:
def classify_with_ollama(title):
    try:
        prompt = f"Does a baby dog appear in this video titled '{title}'?\\nAnswer True or False:"
        response = requests.post('http://localhost:11434/api/generate', json={'model': 'qwen3.5:4b', 'prompt': prompt, 'stream': False})
        result = response.json()['response'].strip().lower()
        return 'true' in result or 'yes' in result
    except:
        return None
print('Function ready')

In [ ]:
results = []
for idx, row in df.iterrows():
    title = row.get('Title', '')
    print(f'[{idx+1}/{len(df)}] {title[:30]}...')
    contains_babydog = classify_with_ollama(title)
    url = row.get('URL', '')
    video_id = url[url.find('v=')+2:].split('&')[0] if 'v=' in url else ''
    results.append({
        'video_id': video_id,
        'title': title,
        'date': row.get('Date', ''),
        'youtube_link': url,
        'contains_babydog': 'True' if contains_babydog else 'False',
        'interesting': '',
        'reviewer': 'Ollama',
        'note': ''
    })
print('Done')

In [ ]:
results_df = pd.DataFrame(results)
output_csv = Path('../../data/export/ai_cleaned-up/ollama.csv')
output_csv.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_csv, index=False)
print(f'Saved {len(results_df)} to {output_csv}')